## Transferring Data from Device manufacturer sites to Influxdb
For flexibility and easy of data manipulation, we transfer the data to influx db.
This will also enable us to have the data downloaded and stored in one place - giving us ownership incase our subscription with the device manufacturer expires.

### Prepare tools for the project
First install the packages and modules necessary to implement the steps of the project.

The documentation for the influxdb_client_3 library can be found [here](https://docs.influxdata.com/influxdb3/cloud-dedicated/reference/client-libraries/v3/python/)

In [18]:
#Install operational packages to help in 
#organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json

#Modules to help interact with influxdb
import os, time
from influxdb_client_3 import InfluxDBClient3, Point


Create a magic function to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [19]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

### Set up influxdb
Prepare the database to receive data from the device manufacturer urls 

In [20]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

#Define some of the arguments required 
#to write the data to the database

token = os.environ.get("INFLUXDB_TOKEN")
org         =   "Air quality - Harrisburg and Philadelphia"
host        =   "https://us-east-1-1.aws.cloud2.influxdata.com"
database    =   "pilot-harrisburg-home"

client = InfluxDBClient3(host=host, token=token, org=org, database=database)

### Upload the data from Devices to influxdb
Make API calls to the device urls and send the data to the influxdb database. Since different manufacturers have different procedures for API calls, we do upload the data in different batches based on the manufacturer

1. QUANTAQ

The guidelines to make API calls to quantaq can be found [here](https://docs.quant-aq.com/software-apis-and-libraries/quantaq-cloud-api)

In [21]:
#Load the api key and device serial numbers from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")
QUANT001_SN = os.environ.get("QUANT001_SN")
QUANT002_SN = os.environ.get("QUANT002_SN")
CLARITY_APIKEY = os.environ.get("CLARITY_APIKEY")
CLARITY_ORGID = os.environ.get("CLARITY_ORGID")


In [161]:
def get_quantaq_data(url_ending_list):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_ending_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending_list:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    #Convert to dictionary for easy manipulation in python
    response_dict = json.loads(response_json.content)

    return response_dict

In [ ]:



all_devices = ['MOD-X-00959','MOD-X-00958']
json_data = get_quantaq_data(['v1','devices','MOD-X-00959','data','?page=10'])
#pd.json_normalize(json_data['data'])
json_data

In [189]:
complete_url_test = "https://api.quant-aq.com/v1/devices/MOD-X-00959/data/?per_page=10000"
response_json = requests.get(complete_url_test, auth=(QUANTAQ_APIKEY,""))
data_dict = json.loads(response_json.content)
df_sample = pd.json_normalize(data_dict['data'])
data_dict['meta']

df_sample


,co,no,no2,o3,pm1,pm10,pm25,raw_data_id,sn,timestamp,...,met.wx_wd,met.wx_ws,met.wx_ws_scalar,model.gas.co,model.gas.no,model.gas.no2,model.gas.o3,model.pm.pm1,model.pm.pm10,model.pm.pm25
0,305.74,29.20,15.76,4.91,10.80,58.87,12.73,20927591,MOD-X-00959,2026-01-09T23:28:45,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
1,304.92,28.13,15.76,4.91,11.11,31.64,13.15,20927589,MOD-X-00959,2026-01-09T23:27:45,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
2,297.95,27.41,15.08,5.03,10.09,37.54,11.93,20927590,MOD-X-00959,2026-01-09T23:26:45,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
3,294.60,26.34,15.08,4.57,10.17,28.37,12.23,20927592,MOD-X-00959,2026-01-09T23:25:45,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
4,293.78,27.77,15.75,5.36,10.20,19.08,11.89,20927301,MOD-X-00959,2026-01-09T23:24:45,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,257.80,0.95,7.55,9.91,17.42,24.67,17.85,20855647,MOD-X-00959,2026-01-09T06:53:02,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
996,254.86,1.72,8.03,9.78,17.74,24.64,18.34,20855358,MOD-X-00959,2026-01-09T06:52:02,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
997,260.12,1.72,8.02,9.77,16.79,38.56,17.33,20855357,MOD-X-00959,2026-01-09T06:51:02,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868
998,258.69,3.26,8.05,9.80,16.67,20.49,17.28,20855356,MOD-X-00959,2026-01-09T06:50:02,...,0.0,0.0,0.0,20508,20509,20510,20511,18867,18869,18868


In [125]:
def upload_to_inlfuxdb(data_frame_to_save, measurement_name, 
                       index_field, client_name, tag_cols_list):
    """
    A function that takes the name of the measurement(table) 
    and one field name to serve as the index and wites them into
    a defined database on influxdb

    Args:
        data_frame_to_save(df)  : The dataframe to be sent to the database
        measurement_name(string) : The measurement or table name to store the dataset
        index_field(string)     : A timefield from the measurement to be used as an index
                                This also serves as the timeseries reference column
        client_name(object)     : The influxdb object with a defined database 
        tag_cols_list (list)    : A list of column names to act as tags

    Return : 
            Prints that it completed the task
    """ 

    #Make the index field to be the index so that it acts as the timestamp
    data_frame_to_save_index = data_frame_to_save.set_index(index_field)
    
    #Write to the database and the let user know that it is written
    client_name.write(record=data_frame_to_save_index, 
                      data_frame_measurement_name=measurement_name,
                      data_frame_tag_columns=tag_cols_list)
    
    print("Done! Check your influxdb to confirm!")



Transform the quantaaq data obtained through the API into a datframe, and send it to the influxdb database for further querrying and vizualization

In [47]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=data_quant002_df, 
                   measurement_name='quantaq-pilot', 
                   index_field='timestamp', 
                   client_name=client)

Done! Check your influxdb to confrim!


In [ ]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Set the UTC timestamp to be the index for direct querrying
data_quant002_df_index = data_quant002_df.set_index('timestamp')

#Define the table name where the data will be saved in influxsb
measurement_name_quantaq = "quantaq-pilot"

#Save the data to influxdb to allow querying and future manipulation
client.write(record=data_quant002_df_index, 
             data_frame_measurement_name = measurement_name_quantaq)

#### 2. Clarity
The general guidelines for accessing clarity data through the api can be found [here](https://api-guide.clarity.io/); These guideline use examples based on POSTMAN - a webtool for obtaining data using APIs. Clarity has a python library for retrieving data found [here](https://github.com/a2gov/clarityio). Here we use the python library.

First install the clarity library

In [8]:
%%skip
pip install clarityio

Make initial connections with the clarity library


In [22]:
import clarityio # Offers methods to connect to the clarity api
from io import StringIO  # Helps to read file as csv and transform to df
api_connection = clarityio.ClarityAPIConnection(api_key=CLARITY_APIKEY, org=CLARITY_ORGID)

Getting the data from the APi before uploading to the database

In [72]:
def create_df_from_date(start_time):

    """
    A function that produces a clarity dataframe from a given to now

    Args:
        start_time(string) : Date and time from to start pulling records
                             In the format of '2026-01-04T00:00:00Z' 
    
    Return:
        latest_clarity_df (dataframe) : A dataframe with data from the given date to now

    """

    #Attemopt to get data from a certain date
    request_body_pilot = {
        'allDatasources': True,
        'outputFrequency': 'hour',
        'format':       'csv-wide',
        'startTime': start_time
        }

    #get the data in form of csv, ocnvert it to a dataframe and 
    # for easy manipulation and saving to the database
    response_wide2 = api_connection.get_recent_measurements(data=request_body_pilot)
    clarity_df = pd.read_csv(StringIO(response_wide2), sep=",")

    #Ensure that only meaningful columns exist to allow 
    # easy data search and manipulation in influxdb
    latest_clarity_df = clarity_df.dropna(axis='columns', how='all')

    return latest_clarity_df


Upload the data from commissioning date to now. This will be the basis from which to contineously update the data

In [158]:
commissioning_time = '2026-01-04T01:00:00Z'
commission_to_jan_26_df= create_df_from_date(start_time=commissioning_time)

In [159]:
commission_to_jan_26_df

,datasourceId,sourceId,sourceType,outputFrequency,startOfPeriod,endOfPeriod,locationLatitude,locationLongitude,no2Conc1HourMean.raw,no2Conc1HourMean.status,...,pm2_5ConcNum1HourMean.value,pm2_5ConcNum24HourRollingMean.raw,pm2_5ConcNum24HourRollingMean.status,pm2_5ConcNum24HourRollingMean.value,relHumidInternal1HourMean.raw,relHumidInternal1HourMean.status,relHumidInternal1HourMean.value,temperatureInternal1HourMean.raw,temperatureInternal1HourMean.status,temperatureInternal1HourMean.value
0,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-09T21:00:00Z,2026-01-09T22:00:00Z,40.315625,-76.895981,33.71,calibration-missing,...,45.75,44.83,sensor-ready,44.83,85.74,sensor-ready,85.74,7.97,sensor-ready,7.97
1,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-09T21:00:00Z,2026-01-09T22:00:00Z,40.264301,-76.851220,24.90,calibration-missing,...,35.91,35.43,sensor-ready,35.43,82.77,sensor-ready,82.77,8.25,sensor-ready,8.25
2,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-09T21:00:00Z,2026-01-09T22:00:00Z,40.315627,-76.895969,29.83,calibration-missing,...,51.02,49.61,sensor-ready,49.61,84.51,sensor-ready,84.51,7.81,sensor-ready,7.81
3,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-09T20:00:00Z,2026-01-09T21:00:00Z,40.315625,-76.895981,28.51,calibration-missing,...,36.54,44.10,sensor-ready,44.10,82.84,sensor-ready,82.84,7.97,sensor-ready,7.97
4,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-09T20:00:00Z,2026-01-09T21:00:00Z,40.264301,-76.851220,27.56,calibration-missing,...,39.10,34.98,sensor-ready,34.98,81.46,sensor-ready,81.46,8.06,sensor-ready,8.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-06T22:00:00Z,2026-01-06T23:00:00Z,40.264301,-76.851220,18.98,calibration-missing,...,59.97,41.21,sensor-ready,41.21,62.02,sensor-ready,62.02,5.24,sensor-ready,5.24
215,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-06T22:00:00Z,2026-01-06T23:00:00Z,37.879027,-122.301929,29.54,calibration-missing,...,74.82,54.09,sensor-ready,54.09,66.22,sensor-ready,66.22,5.11,sensor-ready,5.11
216,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-06T21:00:00Z,2026-01-06T22:00:00Z,37.879027,-122.301929,27.75,calibration-missing,...,71.78,48.80,sensor-ready,48.80,64.51,sensor-ready,64.51,6.08,sensor-ready,6.08
217,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-06T21:00:00Z,2026-01-06T22:00:00Z,40.264301,-76.851220,16.91,calibration-missing,...,61.67,40.39,sensor-ready,40.39,60.68,sensor-ready,60.68,5.90,sensor-ready,5.90


In [154]:
commission_to_jan_26_df[['sourceId','startOfPeriod','endOfPeriod']].sort_values(by='startOfPeriod', ascending=True)

KeyError: "None of [Index(['sourceId', 'startOfPeriod', 'endOfPeriod'], dtype='object')] are in the [columns]"

Use the influxdb query package to get the latest date from the database. Look at the date and ensure that it makes sense. 

In [ ]:
query_date = """SELECT *
                FROM  'clarity-pilot-tags' 
                ORDER BY time DESC LIMIT 1
             """
    
#Use the queery on the influxdb database
table = client.query(query=query_date, language='sql')

#Obtain the date to use as a start date in the uplaoding 
# the most recent data from clarity to influxdb
most_recent_clarity_df = table.to_pandas()
latest_date = most_recent_clarity_df.startOfPeriod[0]

#verify that the date is in the right format and makes sense
print(latest_date)


2026-01-07T20:00:00Z


From the clarity API, get the data from the most recent date of saving in influxdb to now. And then make a dataframe from it which will later be added to the influxdb database

In [75]:
latest_clarity_df = create_df_from_date(start_time=latest_date)

In [128]:
#Sample for this week (01/04/25 - 01/09/25)
week_df = create_df_from_date(start_time="2026-01-04T00:00:00Z")

Upload this dataframe to the influxdb database in the cloud. Ensure that the measurement name and client name in the uplaod function match with what was used to generate the dataframe

In [129]:
#Define the columns to use as tags so that datapoints 
# recorded at exactly the same time can be recorded
columns_tags = ['datasourceId', 'locationLatitude',
                'locationLongitude','sourceId']
#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=week_df, 
                   measurement_name='clarity-pilot-tags', 
                   index_field='endOfPeriod', 
                   client_name=client,
                   tag_cols_list=columns_tags)

Done! Check your influxdb to confirm!
